# Tier-B — Improved MambaLOB architectures (headline novelty)

Phase-2 Tier B. The base MambaLOB is *causal* (left→right scan) and feature-flat. Here we test two
architectural improvements, asking the headline question:

> **Can a better-designed Mamba match or beat TLOB (the transformer SOTA) at lower compute?**

Variants (all in `modeling/models.py`):
- **`bimambalob`** — bidirectional Mamba (`_BiMamba`): each block reads the look-back window forwards *and*
  backwards and averages, so every position sees its full local neighbourhood (still O(L), 2× constant).
- **`convmambalob`** — DeepLOB's Conv+Inception *spatial* front-end → Mamba *temporal* core (replaces DeepLOB's
  sequential LSTM with a selective SSM). Hybrid: CNN inductive bias + O(L) long-range temporal modelling.
- **`biconvmambalob`** — both improvements combined.
- Baselines for context: **`mambalob`** (our base) and **`tlob`** (transformer SOTA target).

All run on the **best feature set from Tier A (`all`, 66 features)** — see Section 3.3 of the report.
Reuses the dual-GPU sharded runner (each shard pinned to one T4 via `CUDA_VISIBLE_DEVICES`).

Setup: enable **GPU T4 x2**; secrets `GH_PAT`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`.
Clean S3 layout: `experiments/architecture/{results,checkpoints,figures}/`.


## 1. Runtime check (expect 2 GPUs)

In [ ]:
import torch, platform
print("Python:", platform.python_version(), "| Torch:", torch.__version__)
n = torch.cuda.device_count()
print(f"GPUs visible: {n}")
for i in range(n):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")
assert n >= 1, "Enable GPU. For 2x speedup choose 'GPU T4 x2'."
if n < 2:
    print("Only 1 GPU -> will run a single shard (still works, no speedup).")

## 2. Get the project code (GH_PAT secret)

In [ ]:
import sys, subprocess, pathlib
REPO_URL = "https://github.com/rajjoseph48/nse-lob-capstone.git"
REPO_DIR = "nse-lob-capstone"
def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    import os
    return os.environ.get(name, "")
GITHUB_TOKEN = _get_secret("GH_PAT")
def add_modeling(base):
    base = pathlib.Path(base)
    for cand in (base / "modeling", base):
        if (cand / "models.py").exists():
            sys.path.insert(0, str(cand.resolve()))
            return str(cand.resolve())
    return None
MODELING_DIR = None
for c in (".", REPO_DIR, "/kaggle/working/" + REPO_DIR, "/content/" + REPO_DIR):
    MODELING_DIR = add_modeling(c)
    if MODELING_DIR:
        break
if not MODELING_DIR:
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)
    MODELING_DIR = add_modeling(REPO_DIR)
print("modeling/ dir:", MODELING_DIR)

## 3. Install deps (Mamba kernel + boto3)

In [ ]:
import subprocess, sys
def pipq(*pkgs): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pipq("boto3")
pipq("ninja", "packaging", "setuptools", "wheel")
pipq("--no-build-isolation", "causal-conv1d")
pipq("--no-build-isolation", "mamba-ssm")
try:
    import mamba_ssm
    print("install step done | mamba-ssm", mamba_ssm.__version__)
except Exception as e:
    print("install step done | mamba-ssm NOT importable -> pure-PyTorch fallback:", repr(e))

## 4. Download NSE data from S3 (shared by both shards)

In [ ]:
import os, boto3, pathlib
os.environ["AWS_ACCESS_KEY_ID"] = _get_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = _get_secret("AWS_SECRET_ACCESS_KEY")
BUCKET, PREFIX, REGION = "lob-capstone-data", "lob-data/dhan/", "ap-south-2"
DATA_DIR_NSE = "nse_data/dhan"; pathlib.Path(DATA_DIR_NSE).mkdir(parents=True, exist_ok=True)
s3 = boto3.client("s3", region_name=REGION)
n = 0
for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX).get("Contents", []):
    if o["Size"] == 0:
        continue
    dst = os.path.join(DATA_DIR_NSE, o["Key"].split("/")[-1])
    if not os.path.exists(dst):
        s3.download_file(BUCKET, o["Key"], dst)
    n += 1
print(f"{n} parquet files in {DATA_DIR_NSE}")

## 5. Build the Tier-B run-list + resume
`GRID` is the experiment list. We compare the improved Mamba variants against the base MambaLOB and the
TLOB SOTA target, all on the `all` feature set, NIFTY + BANKNIFTY, horizons 10 & 100. We pull merged
results from S3 and keep only not-yet-done specs.

In [ ]:
import json, pathlib, pandas as pd
from nbenv import s3_client, s3_pull_area

AREA = "architecture"
MERGED = pathlib.Path("results/architecture.csv")
FEATURE_SET = "all"                                  # best from Tier A
MODELS = ["mambalob", "bimambalob", "convmambalob", "biconvmambalob", "tlob"]

GRID = []
for sym in ["NIFTY", "BANKNIFTY"]:
    for model in MODELS:
        for h in [10, 100]:
            GRID.append({"model": model, "symbol": sym, "feature_set": FEATURE_SET,
                         "horizon": h, "label_scheme": "A", "seed": 0})

_s3 = s3_client()
# Pull the merged CSV AND each shard's incremental CSV (runner pushes shardN.csv
# after every run), so a mid-session kill still resumes at per-run granularity.
pathlib.Path("results").mkdir(exist_ok=True)
s3_pull_area(_s3, MERGED, AREA)
shard_csvs = [pathlib.Path(f"results/shard{i}.csv") for i in range(2)]
for sc in shard_csvs:
    s3_pull_area(_s3, sc, AREA)
done = set()
for f in [MERGED, *shard_csvs]:
    if f.exists():
        d = pd.read_csv(f)
        done |= {(r.model, r.symbol, r.feature_set, int(r.horizon), r.label_scheme, int(r.seed))
                 for r in d.itertuples()}
def _key(s):
    return (s["model"], s["symbol"], s["feature_set"], int(s["horizon"]), s["label_scheme"], int(s["seed"]))
todo = [s for s in GRID if _key(s) not in done]
pathlib.Path("runlist.json").write_text(json.dumps(todo))
print(f"grid={len(GRID)} | done={len(done)} | to run this session={len(todo)}")
todo

## 6. Launch one shard per GPU (concurrent) and stream logs
Each shard is a subprocess pinned to one GPU via `CUDA_VISIBLE_DEVICES`. We tail both logs until done.

In [ ]:
import os, sys, subprocess, time, torch

n_gpus = torch.cuda.device_count()
nshards = max(1, n_gpus)
RUN_SHARD = f"{MODELING_DIR}/run_shard.py"
pathlib.Path("results").mkdir(exist_ok=True)

procs = []
for i in range(nshards):
    env = dict(os.environ); env["CUDA_VISIBLE_DEVICES"] = str(i)
    env["PYTHONUNBUFFERED"] = "1"                       # stream child logs live (no block-buffering)
    logf = f"shard{i}.log"; fh = open(logf, "w")
    cmd = [sys.executable, "-u", RUN_SHARD, "--runlist", "runlist.json", "--shard", str(i),
           "--nshards", str(nshards), "--data-dir", DATA_DIR_NSE, "--out", f"results/shard{i}.csv",
           "--ckpt-dir", "checkpoints/arch", "--area", "architecture", "--epochs", "20"]
    procs.append([subprocess.Popen(cmd, env=env, stdout=fh, stderr=subprocess.STDOUT), fh, logf, 0])
    print(f"launched shard {i} on CUDA_VISIBLE_DEVICES={i}")

while any(p[0].poll() is None for p in procs):
    for p in procs:
        with open(p[2]) as r:
            r.seek(p[3]); chunk = r.read(); p[3] = r.tell()
        if chunk:
            print(chunk, end="")
    time.sleep(10)
for p in procs:                                        # final flush
    p[1].close()
    with open(p[2]) as r:
        r.seek(p[3]); print(r.read(), end="")
print("\nexit codes:", [p[0].poll() for p in procs])

## 7. Merge shard CSVs + push to S3

In [ ]:
import glob, pandas as pd
from nbenv import s3_client, s3_put_area
keys = ["model", "symbol", "feature_set", "horizon", "label_scheme", "seed"]
frames = [pd.read_csv(MERGED)] if MERGED.exists() else []
frames += [pd.read_csv(f) for f in glob.glob("results/shard*.csv")]
alldf = pd.concat(frames, ignore_index=True).drop_duplicates(subset=keys, keep="last") if frames else pd.DataFrame()
alldf.to_csv(MERGED, index=False)
s3_put_area(s3_client(), MERGED, AREA)
print(f"merged {len(alldf)} rows -> {MERGED}  (s3: experiments/{AREA}/results/)")
alldf.sort_values(keys)

## 8. Findings — do the improved architectures beat TLOB at lower compute?
The headline claim is **quality at lower cost**: we read weighted/macro-F1 *and* parameter count and
train time together. A Mamba variant that matches TLOB's F1 with fewer params / less time supports
"we designed a better LOB model".

In [ ]:
import matplotlib.pyplot as plt
res = pd.read_csv(MERGED)
ORDER = ["mambalob", "bimambalob", "convmambalob", "biconvmambalob", "tlob"]

for sym in sorted(res.symbol.unique()):
    sub = res[res.symbol == sym]
    print("=" * 70, f"\n{sym}\n" + "=" * 70)
    piv = sub.pivot_table(index="model", columns="horizon", values="test_weighted_f1").reindex(ORDER)
    print("weighted-F1 by model x horizon:")
    print(piv.round(4).to_string())
    pm = sub.pivot_table(index="model", columns="horizon", values="test_macro_f1").reindex(ORDER)
    print("\nmacro-F1 by model x horizon:")
    print(pm.round(4).to_string())

# Efficiency: params & train time (model-level, averaged over symbol/horizon)
eff = res.groupby("model").agg(
    params=("n_params", "max"),
    avg_train_s=("train_time_s", "mean"),
    avg_wf1=("test_weighted_f1", "mean"),
    avg_mf1=("test_macro_f1", "mean"),
).reindex(ORDER)
print("\nEfficiency vs quality (avg over symbol x horizon):")
print(eff.round({"avg_train_s": 1, "avg_wf1": 4, "avg_mf1": 4}).to_string())

# Quality-vs-compute scatter (the headline plot)
fig, ax = plt.subplots(figsize=(7, 5))
for m in ORDER:
    if m in eff.index and pd.notna(eff.loc[m, "params"]):
        ax.scatter(eff.loc[m, "params"] / 1e3, eff.loc[m, "avg_wf1"], s=90)
        ax.annotate(m, (eff.loc[m, "params"] / 1e3, eff.loc[m, "avg_wf1"]),
                    textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set(xlabel="parameters (k)", ylabel="avg weighted-F1",
       title="Tier-B: quality vs model size (lower-right = better-and-smaller)")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("results/fig_architecture.png", dpi=150, bbox_inches="tight")
s3_put_area(s3_client(), "results/fig_architecture.png", AREA); plt.show()

# vs-TLOB delta summary
tlob_wf1 = res[res.model == "tlob"].groupby(["symbol", "horizon"]).test_weighted_f1.mean()
print("\nDelta weighted-F1 vs TLOB (positive = our variant beats the transformer):")
for m in ["mambalob", "bimambalob", "convmambalob", "biconvmambalob"]:
    mm = res[res.model == m].groupby(["symbol", "horizon"]).test_weighted_f1.mean()
    d = (mm - tlob_wf1).dropna()
    if len(d):
        print(f"  {m:16s} mean Δ={d.mean():+.4f}  (wins {int((d > 0).sum())}/{len(d)} cells)")